# kaggle-vllm 0.1.2 — Fresh-session TP=1 / TP=2 Benchmark

This notebook benchmarks the **published `kaggle-vllm==0.1.2`** runtime on a
completely fresh Kaggle session. It is intentionally self-contained: no source
repository or pre-existing `/kaggle/working` files are required.

Required Kaggle settings:

- Accelerator: **GPU T4 ×2**
- Internet: **On**
- Fresh Kaggle session / fresh kernel
- No attached dataset is required

The notebook first reproduces the validated 0.1.2 runtime bootstrap using the
canonical immutable native artifact, then runs the same controlled TP=1/TP=2
benchmark matrix used by the later benchmark notebook.

Creating this notebook is **not benchmark evidence**. Only a fully executed
Kaggle copy with preserved JSON/log outputs is evidence.

## 1. Verify the fresh Kaggle dual-T4 environment

In [1]:
!pip list > ./kaggle-pip-list.txt

In [2]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

EXPECTED_SDK_VERSION = "0.1.2"
EXPECTED_NATIVE_REPO = "waqasm86/kaggle-vllm-binaries"
EXPECTED_NATIVE_REVISION = "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"
EXPECTED_NATIVE_WHEEL = "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
EXPECTED_NATIVE_SHA256 = "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
EXPECTED_TORCH = "2.10.0+cu128"
EXPECTED_TORCH_CUDA = "12.8"

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("Torch path:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i),
        "capability",
        torch.cuda.get_device_capability(i),
    )

print("NCCL:", ".".join(map(str, torch.cuda.nccl.version())))

assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__ == EXPECTED_TORCH, torch.__version__
assert torch.version.cuda == EXPECTED_TORCH_CUDA, torch.version.cuda
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

TORCH_BEFORE = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
print("Environment validation: PASS")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-ba42d5b9-490a-8330-e589-7fe5ab08871e)
GPU 1: Tesla T4 (UUID: GPU-5add4134-ee77-64cb-f71b-380e7d056ee6)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilat

## 2. Install the exact published 0.1.2 SDK from PyPI

This is the same public package line used by the executed 0.1.2 reset acceptance.
It must not replace Kaggle's system PyTorch.

In [3]:
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "--upgrade",
        "kaggle-vllm[hub]==0.1.2",
    ],
    check=True,
)

import kaggle_vllm

print("kaggle_vllm version:", kaggle_vllm.__version__)
assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION

CLI = shutil.which("kaggle-vllm")
assert CLI, "kaggle-vllm CLI was not installed"

subprocess.run([CLI, "fingerprint"], check=True)
subprocess.run([CLI, "verify-gpus", "--tensor-parallel-size", "2"], check=True)
print("PyPI SDK validation: PASS")

kaggle_vllm version: 0.1.2
{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/usr/local/nvidia/lib64/libcuda.so",
  "cmake_library_path": null
}
PASS: 

## 3. Strict bootstrap into benchmark-owned runtime paths

Every Kaggle session starts fresh, so this notebook performs the full native
runtime delivery before benchmarking. It does not assume anything from a previous
acceptance notebook or session.

The cache is separate from the staged runtime and may be reused only within this
same live session.

In [4]:
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-benchmark-012")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

# Fresh-session invariant: do not silently benchmark through stale notebook-owned state.
for path in (STAGED, OVERLAY, MANIFEST):
    assert not path.exists(), f"Expected fresh benchmark path, but found: {path}"

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

BOOTSTRAP = [
    CLI, "bootstrap", "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

print("=== strict dry-run ===")
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)

print("=== strict bootstrap ===")
subprocess.run(BOOTSTRAP, check=True)

assert STAGED.is_dir(), STAGED
assert OVERLAY.is_dir(), OVERLAY
assert MANIFEST.is_file(), MANIFEST

manifest_text = MANIFEST.read_text(encoding="utf-8")
for expected in (
    EXPECTED_NATIVE_REPO,
    EXPECTED_NATIVE_REVISION,
    EXPECTED_NATIVE_WHEEL,
    EXPECTED_NATIVE_SHA256,
):
    assert expected in manifest_text, f"Missing immutable runtime identity in manifest: {expected}"

print("Manifest:", MANIFEST)
print("Strict bootstrap: PASS")

=== strict dry-run ===
{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "check": "PyTorch",
      "status": "pass",
      "message": "PyTorch: 2.10.0+cu128"
    },
    {
      "check": "PyTorch CUDA",
      "status": "pass",
      "message": "PyTorch CUDA: 12.8"
    },
    {
      "check": "visible GPU count",
      "status": "pass",
      "message": "visible GPU count: 2"
    },
    {
   

Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 19.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 277.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 265.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 402.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 345.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 286.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 338.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 407.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 263.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 329.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## 4. Activate the staged runtime and verify native imports / Torch preservation

In [5]:
from kaggle_vllm import activate_runtime

assert activate_runtime(MANIFEST), f"Could not activate runtime manifest: {MANIFEST}"

import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

print("vLLM version:", vllm.__version__)
print("vLLM path:", vllm.__file__)
print("vllm._C:", vllm._C.__file__)
print("vllm._moe_C:", vllm._moe_C.__file__)
print("vllm.cumem_allocator:", vllm.cumem_allocator.__file__)

staged_resolved = STAGED.resolve()
for module_path in (
    Path(vllm.__file__).resolve(),
    Path(vllm._C.__file__).resolve(),
    Path(vllm._moe_C.__file__).resolve(),
    Path(vllm.cumem_allocator.__file__).resolve(),
):
    assert staged_resolved in module_path.parents, module_path

TORCH_AFTER = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
print("Torch before:", TORCH_BEFORE)
print("Torch after :", TORCH_AFTER)
assert TORCH_AFTER == TORCH_BEFORE, (TORCH_BEFORE, TORCH_AFTER)

print("Native imports: PASS")
print("Torch preservation: PASS")

vLLM version: 0.18.2.dev0+ga26e8dc7f.d20260822
vLLM path: /kaggle/working/kaggle-vllm-benchmark-012/vllm-staged/vllm/__init__.py
vllm._C: /kaggle/working/kaggle-vllm-benchmark-012/vllm-staged/vllm/_C.abi3.so
vllm._moe_C: /kaggle/working/kaggle-vllm-benchmark-012/vllm-staged/vllm/_moe_C.abi3.so
vllm.cumem_allocator: /kaggle/working/kaggle-vllm-benchmark-012/vllm-staged/vllm/cumem_allocator.abi3.so
Torch before: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
Torch after : {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
Native imports: PASS
Torch preservation: PASS


## 5. Create the benchmark harness inside `/kaggle/working`

The benchmark logic is derived from the attached later benchmark notebook, but is
written into this notebook so that no repository source snapshot is required.

In [6]:
BENCHMARK = Path("/kaggle/working/benchmark_kaggle_012.py")
BENCHMARK.write_text('\n"""Kaggle GPU benchmark harness for controlled TP=1/TP=2 comparisons."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport subprocess\nimport threading\nimport time\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Self\n\n\n@dataclass\nclass GPUSample:\n    timestamp: float\n    index: int\n    memory_used_mib: int\n    utilization_percent: int\n\n\nclass GPUMonitor:\n    def __init__(self, interval: float = 0.2) -> None:\n        self.interval = interval\n        self.samples: list[GPUSample] = []\n        self._stop = threading.Event()\n        self._thread = threading.Thread(target=self._collect, daemon=True)\n\n    def _collect(self) -> None:\n        while not self._stop.is_set():\n            command = [\n                "nvidia-smi",\n                "--query-gpu=index,memory.used,utilization.gpu",\n                "--format=csv,noheader,nounits",\n            ]\n            result = subprocess.run(\n                command, capture_output=True, text=True, check=False\n            )\n            if result.returncode == 0:\n                for line in result.stdout.splitlines():\n                    try:\n                        index, memory, utilization = (\n                            int(value.strip()) for value in line.split(",")\n                        )\n                    except (TypeError, ValueError):\n                        continue\n                    self.samples.append(\n                        GPUSample(time.time(), index, memory, utilization)\n                    )\n            self._stop.wait(self.interval)\n\n    def __enter__(self) -> Self:\n        self._thread.start()\n        return self\n\n    def __exit__(self, *_args: object) -> None:\n        self._stop.set()\n        self._thread.join(timeout=max(1.0, self.interval * 3))\n\n    def summary(self) -> list[dict[str, int]]:\n        result = []\n        for index in sorted({sample.index for sample in self.samples}):\n            samples = [sample for sample in self.samples if sample.index == index]\n            result.append(\n                {\n                    "index": index,\n                    "peak_memory_used_mib": max(\n                        sample.memory_used_mib for sample in samples\n                    ),\n                    "peak_utilization_percent": max(\n                        sample.utilization_percent for sample in samples\n                    ),\n                    "last_memory_used_mib": samples[-1].memory_used_mib,\n                }\n            )\n        return result\n\n\ndef _command_output(command: list[str]) -> str | None:\n    try:\n        result = subprocess.run(\n            command, capture_output=True, text=True, check=False, timeout=15\n        )\n    except (OSError, subprocess.SubprocessError):\n        return None\n    return result.stdout.strip() if result.returncode == 0 else None\n\n\ndef result_schema() -> dict[str, Any]:\n    return {\n        "schema_version": 1,\n        "status": "pending_gpu_execution",\n        "configuration": {},\n        "environment": {},\n        "topology": {},\n        "runs": [],\n        "aggregate": {},\n    }\n\n\ndef _metrics(output: Any, elapsed: float) -> dict[str, float | int | None]:\n    prompt_tokens = len(output.prompt_token_ids or ())\n    output_tokens = sum(len(item.token_ids) for item in output.outputs)\n    request_metrics = getattr(output, "metrics", None)\n    first = getattr(request_metrics, "first_token_ts", 0.0) or 0.0\n    arrival = getattr(request_metrics, "arrival_time", 0.0) or 0.0\n    last = getattr(request_metrics, "last_token_ts", 0.0) or 0.0\n    ttft = first - arrival if first > arrival > 0 else None\n    engine_e2e = last - arrival if last > arrival > 0 else None\n    decode = last - first if last > first > 0 else None\n    return {\n        "prompt_tokens": prompt_tokens,\n        "output_tokens": output_tokens,\n        "wall_latency_seconds": elapsed,\n        "engine_latency_seconds": engine_e2e,\n        "time_to_first_token_seconds": ttft,\n        "prefill_tokens_per_second": (\n            prompt_tokens / ttft if ttft and ttft > 0 else None\n        ),\n        "generation_tokens_per_second": (\n            output_tokens / decode if decode and decode > 0 else None\n        ),\n        "end_to_end_tokens_per_second": (\n            output_tokens / elapsed if elapsed > 0 else None\n        ),\n    }\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", default="facebook/opt-125m")\n    parser.add_argument(\n        "--tensor-parallel-size", type=int, choices=(1, 2), required=True\n    )\n    parser.add_argument("--prompt", default="Explain tensor parallel inference.")\n    parser.add_argument("--max-tokens", type=int, default=128)\n    parser.add_argument("--repeats", type=int, default=3)\n    parser.add_argument("--max-model-len", type=int, default=512)\n    parser.add_argument("--gpu-memory-utilization", type=float, default=0.40)\n    parser.add_argument(\n        "--enforce-eager", action=argparse.BooleanOptionalAction, default=True\n    )\n    parser.add_argument(\n        "--disable-custom-all-reduce",\n        action=argparse.BooleanOptionalAction,\n        default=True,\n    )\n    parser.add_argument("--output", type=Path)\n    return parser\n\n\ndef main() -> int:\n    args = build_parser().parse_args()\n    if args.repeats < 1 or args.max_tokens < 1:\n        raise SystemExit("repeats and max-tokens must be positive")\n\n    from vllm import LLM, SamplingParams\n    from kaggle_vllm.environment import collect\n\n    environment = collect()\n    if environment.gpu_count < args.tensor_parallel_size:\n        raise SystemExit("visible GPU count is smaller than tensor parallel size")\n\n    config = {\n        "model": args.model,\n        "tensor_parallel_size": args.tensor_parallel_size,\n        "prompt": args.prompt,\n        "requested_output_tokens": args.max_tokens,\n        "repeats": args.repeats,\n        "max_model_len": args.max_model_len,\n        "gpu_memory_utilization": args.gpu_memory_utilization,\n        "enforce_eager": args.enforce_eager,\n        "disable_custom_all_reduce": args.disable_custom_all_reduce,\n    }\n\n    llm = LLM(\n        model=args.model,\n        tensor_parallel_size=args.tensor_parallel_size,\n        dtype="float16",\n        max_model_len=args.max_model_len,\n        gpu_memory_utilization=args.gpu_memory_utilization,\n        enforce_eager=args.enforce_eager,\n        disable_custom_all_reduce=args.disable_custom_all_reduce,\n    )\n    sampling = SamplingParams(temperature=0.0, max_tokens=args.max_tokens)\n\n    # Warmup is intentionally excluded from measured repeats.\n    llm.generate([args.prompt], sampling)\n\n    runs = []\n    with GPUMonitor() as monitor:\n        for index in range(args.repeats):\n            started = time.perf_counter()\n            outputs = llm.generate([args.prompt], sampling)\n            elapsed = time.perf_counter() - started\n            runs.append({"index": index, **_metrics(outputs[0], elapsed)})\n\n    latencies = [float(run["wall_latency_seconds"]) for run in runs]\n    throughputs = [float(run["end_to_end_tokens_per_second"]) for run in runs]\n\n    payload = result_schema()\n    payload.update(\n        {\n            "status": "executed",\n            "captured_at": datetime.now(timezone.utc).isoformat(),\n            "configuration": config,\n            "environment": asdict(environment),\n            "topology": {\n                "nvidia_smi": _command_output(["nvidia-smi"]),\n                "nvidia_smi_topology": _command_output(["nvidia-smi", "topo", "-m"]),\n                "gpu_monitor": monitor.summary(),\n            },\n            "runs": runs,\n            "aggregate": {\n                "mean_wall_latency_seconds": sum(latencies) / len(latencies),\n                "mean_output_tokens_per_second": sum(throughputs) / len(throughputs),\n            },\n        }\n    )\n\n    rendered = json.dumps(payload, indent=2) + "\\n"\n    if args.output:\n        args.output.write_text(rendered, encoding="utf-8")\n    print(rendered, end="")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
print("Benchmark harness:", BENCHMARK)
print("Harness bytes:", BENCHMARK.stat().st_size)

Benchmark harness: /kaggle/working/benchmark_kaggle_012.py
Harness bytes: 8143


## 6. Controlled benchmark matrix

Each configuration runs in its **own Python process** so GPU allocations and vLLM
workers are released before the next run.

Matrix:

1. TP=1, eager, custom all-reduce disabled
2. TP=2, eager, custom all-reduce disabled
3. TP=1, non-eager, custom all-reduce disabled
4. TP=2, non-eager, custom all-reduce disabled
5. TP=2, eager, custom all-reduce enabled

A failed experimental configuration is retained as evidence and does not erase the
other benchmark runs.

In [7]:
MODEL = "facebook/opt-125m"
RESULTS = Path("/kaggle/working/kaggle-vllm-benchmarks-012")
RESULTS.mkdir(parents=True, exist_ok=True)

matrix = [
    (1, True,  True),
    (2, True,  True),
    (1, False, True),
    (2, False, True),
    (2, True,  False),
]

records = []

for tp, eager, disable_car in matrix:
    name = f"tp{tp}-eager{int(eager)}-disable-car{int(disable_car)}.json"
    output_path = RESULTS / name
    log_path = RESULTS / name.replace(".json", ".log")

    command = [
        sys.executable,
        str(BENCHMARK),
        "--model", MODEL,
        "--tensor-parallel-size", str(tp),
        "--repeats", "3",
        "--max-tokens", "128",
        "--output", str(output_path),
        "--enforce-eager" if eager else "--no-enforce-eager",
        "--disable-custom-all-reduce" if disable_car else "--no-disable-custom-all-reduce",
    ]

    env = os.environ.copy()
    if tp == 1:
        env["CUDA_VISIBLE_DEVICES"] = "0"
    else:
        env.pop("CUDA_VISIBLE_DEVICES", None)

    print("\n" + "=" * 100)
    print("RUN:", name)
    print("=" * 100)

    started = time.time()
    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=env,
    )
    elapsed = time.time() - started

    log_path.write_text(
        "$ " + " ".join(command)
        + "\n\nSTDOUT\n" + completed.stdout
        + "\n\nSTDERR\n" + completed.stderr,
        encoding="utf-8",
    )

    if completed.returncode == 0 and output_path.is_file():
        payload = json.loads(output_path.read_text(encoding="utf-8"))
        payload["wrapper"] = {
            "returncode": completed.returncode,
            "elapsed_seconds": elapsed,
            "log": str(log_path),
        }
        output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        status = payload.get("status", "unknown")
    else:
        payload = {
            "schema_version": 1,
            "status": "failed",
            "configuration": {
                "model": MODEL,
                "tensor_parallel_size": tp,
                "enforce_eager": eager,
                "disable_custom_all_reduce": disable_car,
                "repeats": 3,
                "max_tokens": 128,
            },
            "wrapper": {
                "returncode": completed.returncode,
                "elapsed_seconds": elapsed,
                "log": str(log_path),
            },
            "stdout_tail": completed.stdout[-4000:],
            "stderr_tail": completed.stderr[-8000:],
        }
        output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        status = "failed"

    records.append((name, status, completed.returncode))
    print("STATUS:", status, "returncode:", completed.returncode)

    # Evidence that the just-finished process did not leave active compute workers.
    subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader",
        ],
        check=False,
    )
    time.sleep(3)

print("\nMATRIX COMPLETE")
for record in records:
    print(record)


RUN: tp1-eager1-disable-car1.json
STATUS: executed returncode: 0

RUN: tp2-eager1-disable-car1.json
STATUS: executed returncode: 0

RUN: tp1-eager0-disable-car1.json
STATUS: executed returncode: 0

RUN: tp2-eager0-disable-car1.json
STATUS: executed returncode: 0

RUN: tp2-eager1-disable-car0.json
STATUS: executed returncode: 0

MATRIX COMPLETE
('tp1-eager1-disable-car1.json', 'executed', 0)
('tp2-eager1-disable-car1.json', 'executed', 0)
('tp1-eager0-disable-car1.json', 'executed', 0)
('tp2-eager0-disable-car1.json', 'executed', 0)
('tp2-eager1-disable-car0.json', 'executed', 0)


## 7. Summarize TP=1 / TP=2 results without hiding failures

In [8]:
summary = []

for path in sorted(RESULTS.glob("*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    row = {
        "file": path.name,
        "status": data.get("status"),
        "tp": data.get("configuration", {}).get("tensor_parallel_size"),
        "eager": data.get("configuration", {}).get("enforce_eager"),
        "disable_custom_all_reduce": data.get("configuration", {}).get("disable_custom_all_reduce"),
        "mean_wall_latency_seconds": data.get("aggregate", {}).get("mean_wall_latency_seconds"),
        "mean_output_tokens_per_second": data.get("aggregate", {}).get("mean_output_tokens_per_second"),
        "returncode": data.get("wrapper", {}).get("returncode"),
    }
    summary.append(row)
    print(row)

assert len(summary) == 5, f"Expected five matrix result files, got {len(summary)}"

SUMMARY_PATH = RESULTS / "summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Summary:", SUMMARY_PATH)

{'file': 'tp1-eager0-disable-car1.json', 'status': 'executed', 'tp': 1, 'eager': False, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 0.24877124133331563, 'mean_output_tokens_per_second': 514.5298142497045, 'returncode': 0}
{'file': 'tp1-eager1-disable-car1.json', 'status': 'executed', 'tp': 1, 'eager': True, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 1.725596456333335, 'mean_output_tokens_per_second': 74.1909246995385, 'returncode': 0}
{'file': 'tp2-eager0-disable-car1.json', 'status': 'executed', 'tp': 2, 'eager': False, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 0.33774232400003257, 'mean_output_tokens_per_second': 379.2680137748775, 'returncode': 0}
{'file': 'tp2-eager1-disable-car0.json', 'status': 'executed', 'tp': 2, 'eager': True, 'disable_custom_all_reduce': False, 'mean_wall_latency_seconds': 3.198598029666717, 'mean_output_tokens_per_second': 40.02747859908997, 'returncode': 0}
{'file': 'tp2-eager1-disable-car1.json

## 8. Preserve checksums and create a single evidence archive

In [9]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

RUN_METADATA = RESULTS / "run-metadata.json"
RUN_METADATA.write_text(
    json.dumps(
        {
            "sdk_version": EXPECTED_SDK_VERSION,
            "native_repo": EXPECTED_NATIVE_REPO,
            "native_revision": EXPECTED_NATIVE_REVISION,
            "native_wheel": EXPECTED_NATIVE_WHEEL,
            "native_sha256": EXPECTED_NATIVE_SHA256,
            "runtime_manifest": str(MANIFEST),
            "runtime_manifest_sha256": sha256_file(MANIFEST),
            "torch_before": TORCH_BEFORE,
            "torch_after": TORCH_AFTER,
            "model": MODEL,
            "matrix": [
                {
                    "tensor_parallel_size": tp,
                    "enforce_eager": eager,
                    "disable_custom_all_reduce": disable_car,
                }
                for tp, eager, disable_car in matrix
            ],
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

CHECKSUMS = RESULTS / "SHA256SUMS.txt"
files = sorted(
    p for p in RESULTS.iterdir()
    if p.is_file() and p.name != CHECKSUMS.name
)
CHECKSUMS.write_text(
    "".join(f"{sha256_file(p)}  {p.name}\n" for p in files),
    encoding="utf-8",
)

print(CHECKSUMS.read_text(encoding="utf-8"))

archive_base = Path("/kaggle/working/kaggle-vllm-benchmarks-012-evidence")
archive = shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS)

print("Evidence directory:", RESULTS)
print("Evidence archive:", archive)
print("Archive SHA256:", sha256_file(Path(archive)))

9263608e31a03f10f476479903284b875d444c3cf676aa4e2f431ade8048748b  run-metadata.json
40dadbcf47065ed69db8f8259d8f9a83181f2bc785de1f0a7ba6764afa7b5fa2  summary.json
6ddbd7c1d6716b68cf7358183982f3d0d9c0b23c75dad81b49e3a630d77cd355  tp1-eager0-disable-car1.json
75ee246494db320aa25f3bd1a7cc26c477821f7a75518648da2ac72829795ee8  tp1-eager0-disable-car1.log
778096573c5f7cb94b93394e8ee45b525cfec98ea9dde004273ba7a5ed99b0ae  tp1-eager1-disable-car1.json
2e9e69d012c1909da502333822610a71c9b18a04c8066bd62463f4f8f139690e  tp1-eager1-disable-car1.log
974ed394c816d500ca93af57de957381f44b0b5935247847e9e1d8225e8db6d4  tp2-eager0-disable-car1.json
ccabd87bca55676125064963c7409add23caf8a3293b5a78efd89bf4b7718ad3  tp2-eager0-disable-car1.log
17a0145aba6a0d406560c58e0c809f1e484adc3e1a097f50c4a110069f5dd253  tp2-eager1-disable-car0.json
91a4690b21d3ae45d3a127efb8bf113df61a67254921af8e91b807a8c04924b5  tp2-eager1-disable-car0.log
21eabeb5461c922054dc0a4e40bf9bd5d1e534420c1b802fae93a3905819f270  tp2-eager1-disa

## 9. Final execution gate

`FINAL BENCHMARK MATRIX: EXECUTED` means the five configurations were attempted
and preserved. It does **not** mean every optimization succeeded, and it does not
claim TP=2 is faster than TP=1 until the JSON measurements are reviewed.

In [10]:
subprocess.run(["nvidia-smi"], check=False)

statuses = {row["file"]: row["status"] for row in summary}
print("Recorded statuses:")
print(json.dumps(statuses, indent=2))
print("FINAL BENCHMARK MATRIX: EXECUTED")

Sun Aug 30 14:21:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----